# German Credit Risk Analysis

Explore customer and loan characteristics, then compare tree-based classifiers for predicting the recorded credit-risk category. Keep `german_credit_data.csv` beside this notebook before running it from the repository folder.

## 1. Setup and data preparation

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [ ]:
data_path = Path("german_credit_data.csv")
if not data_path.exists():
    raise FileNotFoundError(
        f"Could not find {data_path}. Place the CSV beside this notebook or update data_path."
    )

df = pd.read_csv(data_path).drop(columns=["Unnamed: 0"], errors="ignore")

for column in ["Saving accounts", "Checking account"]:
    df[column] = df[column].fillna("unknown")

df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

## 2. Exploratory data analysis

In [ ]:
df["Risk"].value_counts()

In [ ]:
df.groupby("Risk")[["Credit amount", "Duration", "Age"]].mean().round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df, x="Risk", y="Credit amount", ax=axes[0])
sns.boxplot(data=df, x="Risk", y="Duration", ax=axes[1])
axes[0].set_title("Credit amount by risk")
axes[1].set_title("Loan duration by risk")
plt.tight_layout()
plt.show()

In [ ]:
pd.crosstab(df["Checking account"], df["Risk"], normalize="index").round(3)

In [ ]:
pd.crosstab(df["Saving accounts"], df["Risk"], normalize="index").round(3)

In [ ]:
purpose_risk = (
    df.groupby(["Purpose", "Risk"])["Credit amount"]
    .sum()
    .unstack(fill_value=0)
)
purpose_risk = purpose_risk.loc[purpose_risk.sum(axis=1).sort_values(ascending=False).index]
plot_colors = ["seagreen" if risk == "good" else "indianred" for risk in purpose_risk.columns]
purpose_risk.plot(kind="bar", figsize=(12, 6), color=plot_colors)
plt.title("Total credit amount by purpose and risk")
plt.xlabel("Purpose")
plt.ylabel("Total credit amount")
plt.xticks(rotation=35, ha="right")
plt.legend(title="Risk")
plt.tight_layout()
plt.show()

In [ ]:
purpose_by_sex = pd.crosstab(df["Purpose"], df["Sex"])
purpose_by_sex.plot(kind="bar", figsize=(12, 5))
plt.title("Credit purpose by sex")
plt.xlabel("Purpose")
plt.ylabel("Number of applicants")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
age_groups = pd.cut(
    df["Age"],
    bins=[-np.inf, 29, 39, np.inf],
    labels=["29 and under", "30-39", "40 and over"],
)
purpose_by_age = pd.crosstab(age_groups, df["Purpose"])
purpose_by_age.plot(kind="bar", figsize=(13, 6))
plt.title("Credit purpose by age group")
plt.xlabel("Age group")
plt.ylabel("Number of applicants")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
risk_by_purpose = pd.crosstab(df["Purpose"], df["Risk"])
bad_to_good_percent = (
    risk_by_purpose["bad"]
    .div(risk_by_purpose["good"].replace(0, np.nan))
    .mul(100)
    .sort_values(ascending=False)
)
ax = bad_to_good_percent.plot(kind="bar", figsize=(12, 5), color="indianred")
ax.bar_label(ax.containers[0], fmt="%.1f%%", padding=3)
plt.title("Bad-to-good applicant count ratio by purpose")
plt.xlabel("Purpose")
plt.ylabel("Bad applicants per 100 good applicants")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
df[["Credit amount", "Duration", "Age"]].corr().round(2)

## 3. Model training and evaluation

The categorical encoder is fitted inside each cross-validation fold to avoid preprocessing leakage. The numeric features are left unscaled because tree-based models do not require feature scaling.

In [ ]:
X = df.drop(columns="Risk")
y = df["Risk"]
categorical_columns = X.select_dtypes(include=["object", "string"]).columns.tolist()
numeric_columns = X.select_dtypes(include="number").columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
        ("numeric", "passthrough", numeric_columns),
    ]
)

In [ ]:
classifiers = {
    "Random forest": RandomForestClassifier(random_state=42),
    "Extra trees": ExtraTreesClassifier(random_state=42),
}
param_grid = {
    "classifier__n_estimators": [300, 450],
    "classifier__max_depth": [None, 10, 12],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, classifier in classifiers.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", classifier),
        ]
    )
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
    results[name] = {
        "best_model": search.best_estimator_,
        "best_params": search.best_params_,
        "cv_accuracy": search.best_score_,
        "test_accuracy": search.score(X_test, y_test),
    }

pd.DataFrame(
    {
        name: {
            "CV accuracy": result["cv_accuracy"],
            "Test accuracy": result["test_accuracy"],
            "Best parameters": result["best_params"],
        }
        for name, result in results.items()
    }
).T.sort_values("Test accuracy", ascending=False)

In [ ]:
for name, result in results.items():
    predictions = result["best_model"].predict(X_test)
    print(f"\n{name}")
    print("Confusion matrix:")
    print(confusion_matrix(y_test, predictions))
    print("Classification report:")
    print(classification_report(y_test, predictions, zero_division=0))

In [ ]:
fig, axes = plt.subplots(len(results), 1, figsize=(12, 5 * len(results)), squeeze=False)

for ax, (name, result) in zip(axes[:, 0], results.items()):
    model = result["best_model"]
    feature_names = model.named_steps["preprocessor"].get_feature_names_out()
    importances = pd.Series(
        model.named_steps["classifier"].feature_importances_,
        index=feature_names,
    ).nlargest(15).sort_values()
    ax.barh(importances.index, importances.values)
    ax.set_title(f"Top feature importances - {name}")
    ax.set_xlabel("Importance")

plt.tight_layout()
plt.show()